# ☕🤖 RoboBarista – unsere neue Artly-inspirierte Kaffeeinsel

**Neue Standard-Testszene** für unsere zukünftige Entwicklung. Der FAIRINO FR5 steht auf einem zentralen Sockel über einer kompakten Insel. Um ihn herum: Mühle, Tamper, Siebträger, Espressomaschine, Milchspender, Milchkanne, **Kunststoffbecher** und Rinser.

**Was du sehen kannst:** ein echtes MuJoCo-Rendering des Raums aus vier Perspektiven, drei Kameras mit erkannten AprilTags, eine **angedeutete Tamper-Absenkung** und eine **Bewegung der Milchkanne** sowie einen Reichweiten- und Sichtbarkeitsbericht.

**Physikalische Grenzen:** Tamper besitzt ein echtes Schubgelenk und einen Positionsmotor. **Der Antrieb wird in einem separaten MuJoCo-`mj_step`-Test bewegt** (55 mm Hub und Rückfahrt, ohne Gravitation und ohne nachgewiesene Anpresskraft gegen Kaffeemehl). In der Videovorschau wird die Hubposition dagegen kinematisch gesetzt. Das Kännchen besitzt ein Freejoint und Masse, wird im Film jedoch **ohne Robotergriff** durch vorgegebene Körperposen bewegt. Die Kunststoffbecher sind vereinfachte transparente Körper ohne reale Wandstärken/Füllstand/Flüssigkeitsphysik. Die Sichtbarkeit von neun festen Stationsmarkern sowie Becher- und oberem Kännchen-Tag wurde für die acht Szenenbilder verifiziert. Der zusätzliche seitliche Kännchen-Tag war noch nicht erkennbar. Eine Sichtbarkeit bei allen späteren Greif- und Bewegungsposen ist **nicht garantiert**. Der FR5 führt in diesem Film noch keine vollständige Zubereitung aus.

Ohne GPU nutzbar. In Colab einfach **Laufzeit → Alle ausführen** wählen.


In [ ]:
# 1. Unser bestehendes GitHub-Projekt laden
import os,subprocess,sys,pathlib
repo=pathlib.Path('/content/robobarista-vision')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1',
       'https://github.com/Coffee-Tobi/robocasa.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo)
print('Projekt:',repo)


In [ ]:
# 2. MuJoCo, AprilTag-Detektor und MP4-Videokodierung
import subprocess,sys,os
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y',
                'libosmesa6-dev','libgl1','libglx-mesa0','libgl1-mesa-dri',
                'ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install',
                '-r','requirements.txt'],check=True)
print('Dependencies OK')


In [ ]:
# 3. Neue Kaffeeinsel rendern und Beweglichkeit prüfen
import os,sys,subprocess
env=os.environ.copy()
env['MUJOCO_GL']='egl' if os.path.exists('/dev/nvidia0') else 'osmesa'
if env['MUJOCO_GL']=='osmesa':env['PYOPENGL_PLATFORM']='osmesa'
cmd=[sys.executable,'-u','-m','robobarista.cell_demo',
     '--frames','8','--fps','5','--width','920','--height','660',
     '--reach','--output','runs/artly_island']
print('Rendering:',env['MUJOCO_GL'],flush=True)
with subprocess.Popen(cmd,env=env,cwd=str(repo),
                      stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                      text=True,errors='replace',bufsize=1) as proc:
    for line in proc.stdout:
        print(line,end='',flush=True)
    exit_code=proc.wait()
if exit_code:raise RuntimeError(f'Rendering exit {exit_code}: Bitte die Ausgabe oben teilen.')


In [ ]:
# 4. Film + Grundriss + Sichtbarkeit ansehen
from IPython.display import display,Video,Image
from pathlib import Path
import json
out=Path(repo)/'runs/artly_island'
video=out/'workcell_demo.mp4'
if video.exists():
    # Kompatibel mit Colab-IPython-Versionen, bei denen
    # Video(filename=...) mit 'stat(NoneType)' abstuerzt:
    try:
        display(Video(str(video), embed=True, width=900))
    except (TypeError, OSError, ValueError) as exc:
        print('MP4-Einbettung nicht verfuegbar:',exc)
        gif=out/'workcell_demo.gif'
        if gif.exists():
            display(Image(filename=str(gif),format='gif'))
        else:
            from IPython.display import FileLink
            display(FileLink(str(video)))
elif (out/'workcell_demo.gif').exists():
    display(Image(filename=str(out/'workcell_demo.gif'),format='gif'))
else:
    print('Kein Film erzeugt. Bitte Fehler aus Schritt 3 ansehen.')
if (out/'layout_topdown.png').exists():
    display(Image(filename=str(out/'layout_topdown.png')))
summary=out/'summary.json'
if summary.exists():
    s=json.loads(summary.read_text())
    print('Stationen:',', '.join(s['stations']))
    print('Tamper im MuJoCo-Modell mit Motor:',s['tamper_actuator_exists'])
    print('Motorischer Hub verifiziert:',s['tamper_motor_diagnostics']['down_success'],
          'Ruecklauf:',s['tamper_motor_diagnostics']['return_success'])
    print('Marker 61..71 (Frames je Marker):',
          {k:s['tag_visibility_by_frame'][str(k)] for k in range(61,72)})
    print('Milchkannen-Marker in den Frames sichtbar:',s['pitcher_tag_visible_frames'])
    print('Beispielhafte IK-Reichweite (keine Kollisionsfreiheit entlang der Wege):')
    for station,check in s['reachability'].items():
        print(f"  {station}: {'IK erreichbar' if check['tested_pose_ik'] else 'NOCH NICHT nachgewiesen'}")


In [ ]:
# 5. Für eine spätere Weiterentwicklung: MP4, Markerzahlen und Zielpositionen sichern
from IPython.display import FileLink,display
import zipfile
p=out/'robobarista_artly_island.zip'
with zipfile.ZipFile(p,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['workcell_preview.png','workcell_last_frame.png',
       'workcell_demo.mp4','workcell_demo.gif','layout_topdown.png',
       'summary.json','visibility.json','reachability.json']:
        f=out/name
        if f.exists():z.write(f,name)
display(FileLink(str(p)))


## Danach

Wenn dir die Anordnung und Sicht der Kameras gefällt, ersetzen wir diese Prototyp-Geräte erst **gezielt** durch passende 3D-Assets, falls der jeweilige Test sie benötigt. Danach folgen:

1. Erreichbarkeit inklusive Greifer-/Geräte- und Kännchenkollisionen über **ganze Bewegungswege**, nicht nur einzelne Stationen.
2. Reale oder virtuelle Greifvorgänge: Siebträger aufnehmen, an Mühle/Tamper fahren, wieder ablegen.
3. Bechertransport, Milchdosierung und später echte Flüssigkeits-/Schaum- und Latte-Art-Modelle.

**Hinweis:** Ein transparenter Übungsbecher ist für Position, Handhabung und erste Gießwege prima. Für die endgültige Latte-Art-Technik sind Bechergeometrie, Wärme und Schaumverhalten relevant; wir müssen das später berücksichtigen.

[Code und Projektplan](https://github.com/Coffee-Tobi/robocasa)